# Screening A - vendor INT8 GEMM oracle (T4)

Does **not** touch glcuda, llama.cpp, or any model. One `nvcc`, ~3 minutes.

The decision this gate makes: is the remaining prefill gap reachable through
GEMM at all? If the summed vendor-GEMM time is a small fraction of the measured
25.2 ms, then rewriting GEMM geometry is bounded well below the 15k target and
the branch should be reclassified before any PTX is written.


In [ ]:
import json, os, pathlib, subprocess, datetime as dt

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)

MEASURED_MS = 25.2088825       # glcuda 244-token prefill, median of 10, oracle-exact
GLCUDA_TPS = 9679.14
LLAMACPP_TPS = 10829.85
TARGET_TPS = 15000.0
PEAK_TOPS = 130.0             # T4 int8 tensor-core peak
PROMPT_TOKENS = 244


def sh(cmd, **kw):
    p = subprocess.run(cmd, capture_output=True, text=True, **kw)
    return p.returncode, p.stdout, p.stderr


rc, out, err = sh(["nvidia-smi", "--query-gpu=name,memory.total",
                   "--format=csv,noheader"])
print(out.strip())
if "T4" not in out:
    raise SystemExit("needs a Tesla T4; got " + repr(out.strip()))


In [ ]:
CU_SRC = r"""
// Vendor INT8 GEMM oracle for the glcuda prefill shapes on Turing.
//
// The question this answers is not "is cuBLAS faster than us" - it is "how much
// of the measured 25.2 ms prefill is GEMM at all". A geometry rewrite is bounded
// by the non-GEMM remainder, so the budget matters more than any single ratio.
#include <cublas_v2.h>
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <vector>

#define CK(x) do { cudaError_t e_=(x); if (e_ != cudaSuccess) { \
  printf("CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e_)); \
  exit(1); } } while (0)
#define CB(x) do { cublasStatus_t s_=(x); if (s_ != CUBLAS_STATUS_SUCCESS) { \
  printf("cuBLAS error %s:%d: %d\n", __FILE__, __LINE__, (int)s_); exit(1); } } while (0)

struct Shape { const char *name; int N; int K; int per_layer; };

// out[M,N] = act[M,K] * W[K,N], issued as the TN layout int8 tensor cores want:
// both operands K-contiguous, so lda = ldb = K (both multiples of 4 here).
static float time_gemm(cublasHandle_t h, int M, int N, int K, int iters) {
  int8_t *A = nullptr, *B = nullptr; int32_t *C = nullptr;
  CK(cudaMalloc(&A, (size_t)K * N));
  CK(cudaMalloc(&B, (size_t)K * M));
  CK(cudaMalloc(&C, (size_t)N * M * sizeof(int32_t)));
  CK(cudaMemset(A, 1, (size_t)K * N));
  CK(cudaMemset(B, 1, (size_t)K * M));
  const int32_t alpha = 1, beta = 0;
  for (int i = 0; i < 10; ++i) {
    CB(cublasGemmEx(h, CUBLAS_OP_T, CUBLAS_OP_N, N, M, K, &alpha,
                    A, CUDA_R_8I, K, B, CUDA_R_8I, K, &beta,
                    C, CUDA_R_32I, N, CUBLAS_COMPUTE_32I,
                    CUBLAS_GEMM_DEFAULT_TENSOR_OP));
  }
  CK(cudaDeviceSynchronize());
  cudaEvent_t t0, t1; CK(cudaEventCreate(&t0)); CK(cudaEventCreate(&t1));
  CK(cudaEventRecord(t0));
  for (int i = 0; i < iters; ++i) {
    CB(cublasGemmEx(h, CUBLAS_OP_T, CUBLAS_OP_N, N, M, K, &alpha,
                    A, CUDA_R_8I, K, B, CUDA_R_8I, K, &beta,
                    C, CUDA_R_32I, N, CUBLAS_COMPUTE_32I,
                    CUBLAS_GEMM_DEFAULT_TENSOR_OP));
  }
  CK(cudaEventRecord(t1));
  CK(cudaEventSynchronize(t1));
  float ms = 0.f; CK(cudaEventElapsedTime(&ms, t0, t1));
  CK(cudaEventDestroy(t0)); CK(cudaEventDestroy(t1));
  CK(cudaFree(A)); CK(cudaFree(B)); CK(cudaFree(C));
  return ms / iters;
}

int main() {
  cudaDeviceProp p; CK(cudaGetDeviceProperties(&p, 0));
  printf("{\"device\":\"%s\",\"sm\":%d.%d,\"sm_count\":%d,\n",
         p.name, p.major, p.minor, p.multiProcessorCount);
  if (p.major != 7 || p.minor != 5) { printf("\"error\":\"not sm_75\"}\n"); return 1; }

  cublasHandle_t h; CB(cublasCreate(&h));
  CB(cublasSetMathMode(h, CUBLAS_TENSOR_OP_MATH));

  // Qwen2.5-0.5B: dim 896, hidden 4864 (gate+up fused = 9728), 24 layers,
  // 14 q heads + 2 kv heads at head_dim 64 -> qkv out = 1152, vocab 151936.
  std::vector<Shape> shapes = {
    {"qkv",       1152,  896, 24},
    {"attn_out",   896,  896, 24},
    {"gate_up",   9728,  896, 24},
    {"ffn_down",   896, 4864, 24},
    {"lm_head", 151936,  896,  1},
  };
  const int Ms[] = {1, 64, 244, 512, 1024, 2048};

  printf("\"results\":[\n");
  bool first = true;
  for (int mi = 0; mi < 6; ++mi) {
    for (size_t si = 0; si < shapes.size(); ++si) {
      const Shape &s = shapes[si];
      int M = Ms[mi];
      int iters = (M * (long long)s.N * s.K > 2000000000LL) ? 30 : 200;
      float ms = time_gemm(h, M, s.N, s.K, iters);
      double ops = 2.0 * M * s.N * s.K;
      if (!first) printf(",\n");
      first = false;
      printf("  {\"m\":%d,\"name\":\"%s\",\"n\":%d,\"k\":%d,\"per_layer\":%d,"
             "\"ms\":%.6f,\"tops\":%.3f}",
             M, s.name, s.N, s.K, s.per_layer, ms, ops / (ms * 1e-3) / 1e12);
    }
  }
  printf("\n]}\n");
  CB(cublasDestroy(h));
  return 0;
}
"""
src = WORK / "gemm_oracle.cu"
src.write_text(CU_SRC, encoding="utf-8")
print("wrote", src, len(CU_SRC), "bytes")


In [ ]:
NVCC = "/usr/local/cuda/bin/nvcc"
if not pathlib.Path(NVCC).exists():
    NVCC = "nvcc"
binp = WORK / "gemm_oracle"
rc, out, err = sh([NVCC, "-O3", "-arch=sm_75", "-o", str(binp), str(src), "-lcublas"])
(RESULTS / "nvcc.log").write_text(
    "returncode=" + str(rc) + chr(10) * 2 + "STDOUT" + chr(10) + out
    + chr(10) * 2 + "STDERR" + chr(10) + err, encoding="utf-8")
if rc:
    raise RuntimeError("nvcc failed (" + str(rc) + "); see nvcc.log" + chr(10) + err[-3000:])
print("compiled ok")


In [ ]:
rc, out, err = sh([str(binp)], env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
(RESULTS / "oracle-raw.log").write_text(
    "returncode=" + str(rc) + chr(10) * 2 + "STDOUT" + chr(10) + out
    + chr(10) * 2 + "STDERR" + chr(10) + err, encoding="utf-8")
if rc:
    raise RuntimeError("oracle failed (" + str(rc) + "); see oracle-raw.log" + chr(10) + err[-2000:])
ORACLE = json.loads(out)
(RESULTS / "oracle.json").write_text(json.dumps(ORACLE, indent=1), encoding="utf-8")
print(ORACLE["device"], "| SMs:", ORACLE["sm_count"], "| rows:", len(ORACLE["results"]))


In [ ]:
rows = ORACLE["results"]
by_m = {}
for r in rows:
    by_m.setdefault(r["m"], []).append(r)


def budget(m, include_lm_head):
    """Summed vendor GEMM time for one prefill pass at batch m."""
    total = 0.0
    for r in by_m[m]:
        if r["name"] == "lm_head" and not include_lm_head:
            continue
        total += r["ms"] * r["per_layer"]
    return total


hdr = "shape".rjust(10) + "N".rjust(8) + "K".rjust(6) + "ms".rjust(10)
hdr += "TOP/s".rjust(9) + "%peak".rjust(8)
print(hdr)
for r in by_m[244]:
    pct = 100.0 * r["tops"] / PEAK_TOPS
    print(r["name"].rjust(10) + str(r["n"]).rjust(8) + str(r["k"]).rjust(6)
          + ("%.4f" % r["ms"]).rjust(10) + ("%.2f" % r["tops"]).rjust(9)
          + ("%.1f%%" % pct).rjust(8))

print()
print("M sweep - vendor efficiency vs batch size (percent of int8 peak):")
names = [r["name"] for r in by_m[244]]
print("shape".rjust(10) + "".join(str(m).rjust(9) for m in sorted(by_m)))
for nm in names:
    line = nm.rjust(10)
    for m in sorted(by_m):
        r = next(x for x in by_m[m] if x["name"] == nm)
        line += ("%.1f%%" % (100.0 * r["tops"] / PEAK_TOPS)).rjust(9)
    print(line)


In [ ]:
LINES = []


def say(s=""):
    print(s)
    LINES.append(s)


say("# Screening A - vendor INT8 GEMM oracle on T4")
say()
say("- device: %s, %d SMs" % (ORACLE["device"], ORACLE["sm_count"]))
say("- glcuda measured prefill: %.2f ms (%.0f tok/s, 244 tokens)"
    % (MEASURED_MS, GLCUDA_TPS))
say("- llama.cpp: %.0f tok/s | target: %.0f tok/s = %.2f ms"
    % (LLAMACPP_TPS, TARGET_TPS, 1000.0 * PROMPT_TOKENS / TARGET_TPS))
say()

# glcuda applies lm_head through `gemv_w` at one row (runner.rs:1238 - prefill
# keeps logits for the last prompt token only), so the budget takes lm_head from
# the M=1 sweep point rather than pretending it is a 244-row GEMM.
G = budget(244, False) + next(x for x in by_m[1] if x["name"] == "lm_head")["ms"]
REM = MEASURED_MS - G
say("## Budget")
say()
say("- vendor GEMM total: **%.2f ms** of the measured %.2f ms (**%.1f%%**)"
    % (G, MEASURED_MS, 100.0 * G / MEASURED_MS))
say("- non-GEMM remainder: **%.2f ms**" % REM)
if REM <= 0:
    say("- remainder is negative: the oracle is slower than our entire prefill, "
        "so this decomposition does not hold. Inconclusive.")
    CEIL_TPS = float("inf")
else:
    CEIL_TPS = 1000.0 * PROMPT_TOKENS / REM
    say("- ceiling with an *infinitely fast* GEMM: **%.0f tok/s** (%.2fx)"
        % (CEIL_TPS, CEIL_TPS / GLCUDA_TPS))
    say("- beats llama.cpp on GEMM alone: **%s**"
        % ("YES" if CEIL_TPS >= LLAMACPP_TPS else "NO"))
    say("- 15k reachable through GEMM alone: **%s**"
        % ("YES" if CEIL_TPS >= TARGET_TPS else "NO"))
say()

# The sub-slab question, priced directly: glcuda issues 64 output-row slabs
# today (runner.rs:772, an explicit Phase A deferral). If the vendor is much
# slower at M=64 than at M=244, that deferral has a measurable price.
say("## Sub-slab price (M=64 vs M=244)")
say()
for nm in [r["name"] for r in by_m[244]]:
    a = next(x for x in by_m[64] if x["name"] == nm)["tops"]
    b = next(x for x in by_m[244] if x["name"] == nm)["tops"]
    say("- `%s`: %.1f%% peak at M=64 vs %.1f%% at M=244 (%.2fx)"
        % (nm, 100.0 * a / PEAK_TOPS, 100.0 * b / PEAK_TOPS, b / max(a, 1e-9)))
say()

say("## Verdict")
say()
if REM > 0 and CEIL_TPS < TARGET_TPS:
    say("GEMM alone cannot reach the target: the non-GEMM remainder already "
        "exceeds the target time budget. Reclassify 15k, or find out what the "
        "remainder contains, before writing any GEMM geometry.")
elif G / MEASURED_MS < 0.35:
    say("GEMM is a minority of the wall clock. A geometry rewrite is bounded "
        "well below its isolated speedup; profile the remainder first.")
else:
    say("GEMM dominates the wall clock and the vendor is materially faster - "
        "the geometry branch is live. Proceed to Screening B (geometry only, "
        "Q8 arithmetic held constant).")

(RESULTS / "REPORT.md").write_text(chr(10).join(LINES), encoding="utf-8")


In [ ]:
import shutil

stamp = dt.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
shutil.make_archive(str(WORK / ("gemm-oracle-" + stamp)), "zip", RESULTS)
print("archived: gemm-oracle-" + stamp + ".zip")
